### Database Setup
Initialize an in-memory SQLite database and seed it with `employees` and `departments` tables.

In [5]:
import sqlite3
import pandas as pd

# Initialize database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create tables
cursor.execute('''
CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT
);
''')

cursor.execute('''
CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT,
    department_id INTEGER,
    salary REAL,
    hire_date DATE,
    manager_id INTEGER,
    FOREIGN KEY(department_id) REFERENCES departments(department_id)
);
''')

# Populate data
departments_data = [
    (1, 'HR'),
    (2, 'Engineering'),
    (3, 'Marketing'),
    (4, 'Sales')
]
cursor.executemany('INSERT INTO departments VALUES (?, ?)', departments_data)

employees_data = [
    (101, 'John', 'Doe', 1, 60000, '2020-01-15', 103),
    (102, 'Jane', 'Smith', 2, 95000, '2019-03-10', 105),
    (103, 'Bob', 'Johnson', 1, 80000, '2018-05-23', None),
    (104, 'Alice', 'Brown', 2, 120000, '2021-07-01', 105),
    (105, 'Charlie', 'Davis', 2, 150000, '2015-11-12', None),
    (106, 'Eva', 'Wilson', 3, 70000, '2022-02-15', 103),
    (107, 'Frank', 'Thomas', 4, 85000, '2021-09-18', 105),
    (108, 'Grace', 'Lee', 3, 70000, '2022-05-01', 103)
]
cursor.executemany('INSERT INTO employees VALUES (?, ?, ?, ?, ?, ?, ?)', employees_data)
conn.commit()

def run_query(sql_query):
    """Helper function to execute and display SQL queries as a pandas DataFrame"""
    return pd.read_sql_query(sql_query, conn)

### Question 1
**Prompt:** Write a SQL query to retrieve the first name, last name, and salary of all employees who earn more than $80,000.

In [6]:
query = """
SELECT first_name, last_name, salary
FROM employees
WHERE salary > 80000;
"""
run_query(query)

,first_name,last_name,salary
0,Jane,Smith,95000.0
1,Alice,Brown,120000.0
2,Charlie,Davis,150000.0
3,Frank,Thomas,85000.0


### Question 2
**Prompt:** Write a SQL query to list all employees along with their corresponding department names.

In [7]:
query = """
SELECT e.first_name, e.last_name, d.department_name
FROM employees e
LEFT JOIN departments d ON e.department_id = d.department_id;
"""
run_query(query)

,first_name,last_name,department_name
0,John,Doe,HR
1,Jane,Smith,Engineering
2,Bob,Johnson,HR
3,Alice,Brown,Engineering
4,Charlie,Davis,Engineering
5,Eva,Wilson,Marketing
6,Frank,Thomas,Sales
7,Grace,Lee,Marketing


### Question 3
**Prompt:** Write a SQL query to find the average salary and total count of employees in each department.

In [8]:
query = """
SELECT d.department_name,
       AVG(e.salary) AS average_salary,
       COUNT(e.emp_id) AS total_employees
FROM employees e
JOIN departments d ON e.department_id = d.department_id
GROUP BY d.department_name;
"""
run_query(query)

,department_name,average_salary,total_employees
0,Engineering,121666.666667,3
1,HR,70000.000000,2
2,Marketing,70000.000000,2
3,Sales,85000.000000,1


### Question 4
**Prompt:** Write a SQL query to identify all employees who do not report to anyone (i.e., they do not have a manager).

In [9]:
query = """
SELECT first_name, last_name
FROM employees
WHERE manager_id IS NULL;
"""
run_query(query)

,first_name,last_name
0,Bob,Johnson
1,Charlie,Davis


### Question 5
**Prompt:** Write a SQL query to find all employees who earn more than the overall average salary of the entire company.

In [10]:
query = """
SELECT first_name, last_name, salary
FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees);
"""
run_query(query)

,first_name,last_name,salary
0,Jane,Smith,95000.0
1,Alice,Brown,120000.0
2,Charlie,Davis,150000.0


### Question 6
**Prompt:** Write a SQL query to list each employee's name along with their manager's name in a single row.

In [11]:
query = """
SELECT e.first_name AS employee_name,
       m.first_name AS manager_name
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.emp_id;
"""
run_query(query)

,employee_name,manager_name
0,John,Bob
1,Jane,Charlie
2,Bob,None
3,Alice,Charlie
4,Charlie,None
5,Eva,Bob
6,Frank,Charlie
7,Grace,Bob


### Question 7
**Prompt:** Write a SQL query to rank employees by salary within their respective departments.

In [12]:
query = """
SELECT department_id,
       first_name,
       salary,
       RANK() OVER(PARTITION BY department_id ORDER BY salary DESC) as salary_rank
FROM employees;
"""
run_query(query)

,department_id,first_name,salary,salary_rank
0,1,Bob,80000.0,1
1,1,John,60000.0,2
2,2,Charlie,150000.0,1
3,2,Alice,120000.0,2
4,2,Jane,95000.0,3
5,3,Eva,70000.0,1
6,3,Grace,70000.0,1
7,4,Frank,85000.0,1


### Question 8
**Prompt:** Write a SQL query to identify duplicate salary values in the organization and find how many times they occur.

In [13]:
query = """
SELECT salary, COUNT(*)
FROM employees
GROUP BY salary
HAVING COUNT(*) > 1;
"""
run_query(query)

,salary,COUNT(*)
0,70000.0,2


### Question 9
**Prompt:** Write a SQL query to retrieve all details for employees whose first name starts with 'J'.

In [14]:
query = """
SELECT first_name, last_name, salary
FROM employees
WHERE first_name LIKE 'J%';
"""
run_query(query)

,first_name,last_name,salary
0,John,Doe,60000.0
1,Jane,Smith,95000.0


### Question 10
**Prompt:** Write a SQL query using a Common Table Expression (CTE) to retrieve the department name and total payroll of the department that has the highest cumulative salary expenditure.

In [15]:
query = """
WITH DeptPayroll AS (
    SELECT d.department_name,
           SUM(e.salary) AS total_payroll
    FROM employees e
    JOIN departments d ON e.department_id = d.department_id
    GROUP BY d.department_name
)
SELECT department_name, total_payroll
FROM DeptPayroll
ORDER BY total_payroll DESC
LIMIT 1;
"""
run_query(query)

,department_name,total_payroll
0,Engineering,365000.0


### Question 11
**Prompt:** Write a SQL query to insert a new department named 'Finance' with `department_id = 5` into the `departments` table, and then verify the insertion.

In [16]:
# Perform INSERT
cursor.execute("""
INSERT INTO departments (department_id, department_name)
VALUES (5, 'Finance');
""")
conn.commit()

# Display the updated departments list to show output
run_query("SELECT * FROM departments;")

,department_id,department_name
0,1,HR
1,2,Engineering
2,3,Marketing
3,4,Sales
4,5,Finance


### Question 12
**Prompt:** Write a SQL query to update the salary of the employee named 'John Doe' (emp_id = 101) to $65,000, and display their updated details.

In [17]:
# Perform UPDATE
cursor.execute("""
UPDATE employees
SET salary = 65000
WHERE emp_id = 101;
""")
conn.commit()

# Display the updated record
run_query("SELECT first_name, last_name, salary FROM employees WHERE emp_id = 101;")

,first_name,last_name,salary
0,John,Doe,65000.0


### Question 13
**Prompt:** Write a SQL query to delete the employee record for 'Frank Thomas' (emp_id = 107) from the `employees` table, and show the remaining team members in his department (department_id = 4).

In [18]:
# Perform DELETE
cursor.execute("""
DELETE FROM employees
WHERE emp_id = 107;
""")
conn.commit()

# Verify if any employees remain in Department 4 (Sales)
run_query("SELECT * FROM employees WHERE department_id = 4;")

,emp_id,first_name,last_name,department_id,salary,hire_date,manager_id


### Question 14
**Prompt:** Write a SQL query to alter the `employees` table by adding a new column named `email` (TEXT data type), and verify the schema.

In [19]:
# Perform ALTER TABLE
cursor.execute("""
ALTER TABLE employees
ADD COLUMN email TEXT;
""")
conn.commit()

# Display table columns to verify the modification
run_query("PRAGMA table_info(employees);")

,cid,name,type,notnull,dflt_value,pk
0,0,emp_id,INTEGER,0,None,1
1,1,first_name,TEXT,0,None,0
2,2,last_name,TEXT,0,None,0
3,3,department_id,INTEGER,0,None,0
4,4,salary,REAL,0,None,0
5,5,hire_date,DATE,0,None,0
6,6,manager_id,INTEGER,0,None,0
7,7,email,TEXT,0,None,0
